# Training & Tuning

## Trainer Module

### Uitilities

In [12]:
def get_model_path(suffix=""):
	return MODEL_DIR / (suffix + "_ckpt.pth")

def save_checkpoint(model, opt, path=None, suffix=""):
	if path==None: path=get_model_path(suffix)
	ck = {'model_state': model.state_dict(), 'opt_state': opt.state_dict()}
	torch.save(ck, path)
	print("Saved checkpoint to", path)

def load_checkpoint(model, opt, load_epoch=0, best_epoch=-1, alpha=0.5, map_location='cpu'):
	path = get_model_path(f"Epoch{load_epoch}")
	ck = torch.load(path, map_location=map_location)
	opt.load_state_dict(ck['opt_state'])

	if best_epoch == -1:
		model.load_state_dict(ck['model_state'])	
		print("Loaded checkpoint from:", path)
	else:
		path_best = get_model_path(f"Epoch{best_epoch}")
		ck2 = torch.load(path_best, map_location=map_location)
		sd1 = ck["model_state"]
		sd2 = ck2["model_state"]
		merged_sd = {}
		for k in sd1.keys():	# assumes same keys/shapes in both checkpoints
			merged_sd[k] = (1-alpha) * sd1[k] + alpha * sd2[k]
		model.load_state_dict(merged_sd)
		print(f"Loaded checkpoint avgs(ratio={alpha}) from: \n1.{path} \n2.{path_best}")
	return path

class SuppressPrints:
	def __enter__(self):
		self._original_stdout = sys.stdout
		sys.stdout = open(os.devnull, 'w')

	def __exit__(self, exc_type, exc_val, exc_tb):
		sys.stdout.close()
		sys.stdout = self._original_stdout

def check_data_exists(file):
	entries = []
	if os.path.exists(file):
		entries = list(csv.DictReader(open(file)))
		return True, entries
	return False, entries

def truncate_file(file, pre_epochs):
	file_exists, file_entries = check_data_exists(file)
	if file_exists and len(file_entries)>0:
		last_entry = file_entries[-1] # skip header auto, read last epoch
		if int(last_entry['epoch']) > pre_epochs:
			print(f"Handling {os.path.basename(file)}, rewritting existing data upto {pre_epochs}-epochs only!")
			with open(file, newline='') as f:
				reader = csv.DictReader(f)
				fieldnames = reader.fieldnames
				try: rows = [r for r in reader if int(r.get('epoch', -1)) <= pre_epochs]
				except: rows = []
			with open(file, 'w', newline='') as f:
				writer = csv.DictWriter(f, fieldnames=fieldnames)
				writer.writeheader()
				writer.writerows(rows)

### Main fn

In [13]:
# Full-scale robust training loop
def Trainer(model, optimizer, loss_fn, cfg, prev_best_epoch=-1, blend=0.5):
	model.to(cfg.device) # model path doesn't exist, revert back to default
	make_beta_schedule(cfg.max_T, device=cfg.device)
	#make_beta_schedule(cfg.max_T, cfg.beta_start, cfg.beta_end, device=cfg.device)

	pre_epochs = 0 # read no of loss entries in loss csv file
	loss_file = MODEL_DIR / "Loss.csv"
	config_file = MODEL_DIR / "Config.csv"
	loss_file_exists, loss_file_entries = check_data_exists(loss_file)
	config_file_exists = os.path.exists(config_file)

	if loss_file_exists and len(loss_file_entries)>0:        
		last_entry = loss_file_entries[-1] # skip header auto, read last epoch
		pre_epochs = int(last_entry['epoch'])
		while pre_epochs > 0:
			try:
				load_checkpoint(model, optimizer, pre_epochs, prev_best_epoch, blend, map_location=cfg.device)
				break
			except FileNotFoundError:
				print(f"WARNING: loading ckpt for ep-{pre_epochs} not found. Trying ep-{pre_epochs - 1}.")
				pre_epochs -= 1 # check for prev models
				continue
			except Exception as e:
				raise Exception(e)
	
	truncate_file(loss_file, pre_epochs)
	truncate_file(config_file, pre_epochs)

	total_batchs = 0 # total no of batches procesed
	model.train()
	st_ep = pre_epochs+1
	ed_ep = pre_epochs + cfg.epochs + 1
	start_time = time.time()
	
	print(f"Trainer initialized [Parameters count = {sum(p.numel() for p in model.parameters())}] for given configuration:")
	pprint.pprint(cfg); print()
	for epoch in range(st_ep, ed_ep):
		loss = 0; val_loss = 0; metrics={'epoch':epoch, 'step':0}
		pbar = tqdm.tqdm(train_loader, desc=f"n_spl={metrics['step']}, Ep[{epoch}]")

		for step, batch in enumerate(pbar):
			optimizer.zero_grad()
			x0 = batch['m3d'].to(cfg.device)
			p2d = batch['p2d'].to(cfg.device)
			B,S,_ = x0.shape
			metrics['step']  = step*train_loader.batch_size + B
			step += 1

			# sample random timesteps for diffusion training (standard & PTSS)
			if cfg.ptss:
				t = probabilistic_timestep_sampling(B, S, cfg.diffsteps, cfg.pts_prob).to(cfg.device)
			else:
				t = torch.randint(0, cfg.diffsteps, (B,), device=cfg.device)
			xt = forward_diffusion_sample(x0, t)
			
			try:
				# forward + loss
				with SuppressPrints():
					out = model(xt, t, p2d)
					loss, loss_dict = loss_fn(out, x0, t) # compute losses
					
				# quick forward NaN check
				if torch.isnan(out).any() or torch.isinf(out).any():
					raise RuntimeError(f"NaN/Inf in model output! stats: NaNs={torch.isnan(out).sum().item()}, Infs={torch.isinf(out).sum().item()}, maxabs={out.abs().max().item()}")
				
				# guard against huge loss
				if not torch.isfinite(loss):
						raise RuntimeError("Non-finite loss")
				loss.backward() # backward
				
			except Exception as e:                
				if cfg.skip_invalid:
					print(f"[Step {step}] WARNING: Error raised: {e}; skipping this batch!")
					continue
				else:
					raise RuntimeError(f"[Step {step}] Error raised: {e}")
			
			# check for NaN in grads
			'''
			grad_nan = any((p.grad is not None) and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()) for p in model.parameters())
			if grad_nan:
				print(f"[Step {step}] WARNING: NaN/Inf gradients detected; skipping optimizer.step() and zero grads.")
				optimizer.zero_grad() # reset
				continue
			'''
			optimizer.step()
			total_steps = total_batchs + step

			# loss accumulation
			upd_loss = {'total':float(loss.item())}
			upd_loss.update(loss_dict)
			upd_val_loss = {k:'-' for k,v in upd_loss.items()} # initialize empty

			# periodic validation (random-t eval)
			if (total_steps % cfg.val_interval == 0) or step == len(train_loader):
				model.eval()
				# quick val: single batch
				with torch.no_grad():
					try:
						vb = next(iter(val_loader))
					except StopIteration:
						vb = None
					if vb is not None:
						x0v = vb['m3d'].to(cfg.device)
						p2dv = vb['p2d'].to(cfg.device)
						B,S,_ = x0v.shape
						if cfg.ptss:
							tv = probabilistic_timestep_sampling(B, S, cfg.diffsteps, cfg.pts_prob).to(cfg.device)
						else:
							tv = torch.randint(0, cfg.diffsteps, (B,), device=cfg.device)
						xtv = forward_diffusion_sample(x0v, tv)
						with SuppressPrints():
							predv = model(xtv, tv, p2dv)
							val_loss, val_loss_dict = loss_fn(predv, x0v, tv)
				model.train()
				# val loss accumulation update
				upd_val_loss['total']   = float(val_loss.item())
				upd_val_loss.update(val_loss_dict)
			
			# logging
			pbar.set_description(f"n_spl={metrics['step']}, Ep[{epoch}]")
			pbar.set_postfix_str(f"loss={float(loss.item()):.6f}, L_std={loss_dict['pred_std']:.6f}, L_simple={loss_dict['simple']:.6f}, " + \
								 f"L_joints={loss_dict['joints']:.6f}, L_vel={loss_dict['vel']:.6f}, L_foot={loss_dict['foot']:.6f}, "+ \
								 f"L_temporal={loss_dict['temporal']:.6f}, L_diversity={loss_dict['diversity']:.6f}, prev_val_loss={val_loss:.6f}")

			# checkpoint (-1 to disable checkpoint saving)
			if cfg.ckpt_interval >= 0 and total_steps % cfg.ckpt_interval == 0:
				with SuppressPrints():
					save_checkpoint(model, optimizer, suffix=f"Ep{epoch}_Step{metrics['step']}")

			# save loss metrics
			metrics.update({'L_'+k:v for k,v in upd_loss.items()})
			metrics.update({'VL_'+k:v for k,v in upd_val_loss.items()})
		
			with open(loss_file, mode='a', newline='') as file:
				writer = csv.DictWriter(file, fieldnames=list(metrics.keys()))
				if not loss_file_exists:
					writer.writeheader() # Write header only if file was empty
					loss_file_exists = True
				writer.writerow(metrics) # Write the metrics for this mini-epoch

		# epoch end checkpoint
		total_batchs = total_steps
		with SuppressPrints():
			save_checkpoint(model, optimizer, suffix=f"Epoch{epoch}")
		
		config_data = {'epoch':epoch}
		config_data.update(cfg.__dict__)
		with open(config_file, mode='a', newline='') as file:
			writer = csv.DictWriter(file, fieldnames=list(config_data.keys()))
			if not config_file_exists:
				writer.writeheader() # Write header only if file was empty
				config_file_exists = True
			writer.writerow(config_data) # Write the config for this epoch

	total_time = time.time() - start_time
	print("Training finished. Total time:", total_time)


## Setup

In [ ]:
import sys, os, time, glob, tqdm, csv
from pathlib import Path
import numpy as np
import torch, pprint
from torch.utils.data import DataLoader
from types import SimpleNamespace
from adan_pytorch import Adan

# directories - repo cfg
DATA_DIR = Path.cwd() / "datasets"
OUT_DIR = Path.cwd() / "outputs"
processed_data_dir = DATA_DIR / "AIST/processed" 

# saved onto .py files after prev notebook runs for reusability
from include.dataset import ViMoDataset
from include.model.vimo_type2 import ViMoFrameWorkv1, ViMoFrameWorkv2
from include.loss_utils import compute_losses, compact_optim_str
from include.ddpm_utils import make_beta_schedule, forward_diffusion_sample, probabilistic_timestep_sampling


# configuration
cfg = SimpleNamespace()
cfg.model_version	= 1		# choose model version: v1 or v2
run_id = f"_v{cfg.model_version}(sq) PTSS" # similuation id for saving in seperate folder of same model_optim (change to non existing id for fresh start)
#run_id = f"_v{cfg.model_version}(sq) PTSS lowbeta"

# global params
cfg.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
files = sorted(glob.glob(os.path.join(processed_data_dir, "*.npz")))

# model params
cfg.motion_dim 		= 151
cfg.pose_dim 		= 17*3
cfg.max_T 			= 1000
cfg.embed_dim 		= 256
cfg.n_heads 		= 8     # no. of attention heads
cfg.m_nlayers 		= 3   	# no. of motion denoiser layers
cfg.p_nlayers 		= 2   	# no. of pose transformer layers
cfg.cond_drop 		= 0.25	# condition drop probability

# beta scheduler params (def 1e-4 to 0.02)
#cfg.beta_start		= 1e-5
#cfg.beta_end		= 0.01

# optimizer params
cfg.learning_rate	= 0.0001 	# defualt: 0.0001
cfg.weight_decay	= 0.02		# default: 0.02

# training params
cfg.batch_size 		= 128	# batch size for training
cfg.epochs 			= 2   	# no. of training epochs
cfg.val_interval 	= 5		# batch interval to perform validation
cfg.ckpt_interval	= -1	# if>0 saves model weights as resp batch intervals
cfg.train_views		= 5		# total 9 cam views, taking 6 views for training, 3 views for validation
cfg.skip_invalid	= False # skip invalid batches
cfg.ptss			= True	# to apply Probabilistic Timestep Sampling Strategy (PTSS) frame wise or not
cfg.pts_prob		= 0.4 	# prob at which resp batch is taken for random timesteps for each frame
cfg.diffsteps 		= 1000 # to make learn on closer time more

# loss params  default (0.05, 0.01, 0.02, 0, 0)
cfg.lambda_joints	= 0.1
cfg.lambda_vel		= 0.01
cfg.lambda_foot    	= 0.02
cfg.lambda_temporal	= 0
cfg.lambda_diversity= 0
cfg.snr_type		= 'sq/sq'		# snr weighting type (refer ddpm_utils.py)
cfg.snr_cap			= 5.0		# snr weight capping
cfg.snr_norm		= True		# snr weight batch normalization
cfg.snr_ratio		= 0.5		# snr blending ratio for averaging methods

## Final Runners

In [15]:
# =-=-=-=-=-=-=-=-= choose model version here =-=-=-=-=-=-=-=-=
if cfg.model_version == 1:
	model = ViMoFrameWorkv1(motion_dim=cfg.motion_dim, pose_dim=cfg.pose_dim, max_T=cfg.max_T, embed_dim=cfg.embed_dim,
						    n_heads=cfg.n_heads, m_nlayers=cfg.m_nlayers, p_nlayers=cfg.p_nlayers, cond_drop=cfg.cond_drop)
elif cfg.model_version == 2:
	model = ViMoFrameWorkv2(motion_dim=cfg.motion_dim, pose_dim=cfg.pose_dim, max_T=cfg.max_T, embed_dim=cfg.embed_dim,
						    n_heads=cfg.n_heads, m_nlayers=cfg.m_nlayers, p_nlayers=cfg.p_nlayers, cond_drop=cfg.cond_drop)
else:
	raise ValueError(f"Invalid model_version {cfg.model_version} specified in config!")

# =-=-=-=-=-=-=-=-= choose optimizer here =-=-=-=-=-=-=-=-=
#optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
optimizer = Adan(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

# =-=-=-=-=-=-=-=-= other initalizations based on config =-=-=-=-=-=-=-=-=
loss_fn = lambda m_pred, m_gt, t: compute_losses(m_pred, m_gt, t, lambda_joints=cfg.lambda_joints, lambda_vel=cfg.lambda_vel, lambda_foot=cfg.lambda_foot, 
												 lambda_temporal=cfg.lambda_temporal, lambda_diversity=cfg.lambda_diversity,
											     snr_type=cfg.snr_type, snr_cap=cfg.snr_cap, snr_norm=cfg.snr_norm, snr_ratio=cfg.snr_ratio)

t_ds, v_ds = ViMoDataset.get_train_val_pair(files, t_views=cfg.train_views, conf_thresh=0.2)
train_loader = DataLoader(t_ds, batch_size=cfg.batch_size, shuffle=True, pin_memory=True)
val_loader   = DataLoader(v_ds, batch_size=cfg.batch_size, shuffle=True, pin_memory=True)

MODEL_DIR = OUT_DIR / f"{model.compact_name}_{compact_optim_str(optimizer)}/run{run_id}/models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

cfg.optim = optimizer.__class__.__name__

In [ ]:
Trainer(model, optimizer, loss_fn, cfg, 61, 0.2)

Loaded checkpoint from: d:\STUDIES\MTech\#MTP\codes\vimo\outputs\ViMo(m=151,p=51,t=1000,d=256,h=8,ml=3,pl=2,c=0.25)_Adan(wd=0.02)\run_v1(sq) PTSS\models\Epoch114_ckpt.pth
Trainer initialized [Parameters count = 5225111] for given configuration:
namespace(model_version=1,
          device=device(type='cuda'),
          motion_dim=151,
          pose_dim=51,
          max_T=1000,
          embed_dim=256,
          n_heads=8,
          m_nlayers=3,
          p_nlayers=2,
          cond_drop=0.25,
          learning_rate=0.0001,
          weight_decay=0.02,
          batch_size=128,
          epochs=6,
          val_interval=5,
          ckpt_interval=-1,
          train_views=5,
          skip_invalid=False,
          ptss=True,
          pts_prob=0.4,
          diffsteps=1000,
          lambda_joints=0.1,
          lambda_vel=0.01,
          lambda_foot=0.02,
          lambda_temporal=0,
          lambda_diversity=0,
          snr_type='sq/sq',
          snr_cap=5.0,
          snr_norm=T

n_spl=21775, Ep[120]: 100%|██████████| 171/171 [10:25<00:00,  3.66s/it, loss=0.018777, L_std=0.106264, L_simple=0.014713, L_joints=0.040226, L_vel=0.002327, L_foot=0.000907, L_temporal=0.003982, L_diversity=0.557389, prev_val_loss=0.011573]

Training finished. Total time: 3715.3693845272064


### History (Changes):
- Run 1.Problems on small Model(d=128,n=4):
	- Epoch[1 - 5]: `lambda_joints=0, lambda_vel=0, lambda_foot=0`; i.e. pure ddpm loss formulation
	- L_pred_std: Drops to 0.003-0.008 => CATASTROPHIC COLLAPSE! i.e. Predicted motions become ~99.5% static!
	- Your model predictions have almost zero temporal variation
	- All predicted frames are nearly identical (static T-pose or mean pose)
	- This is the "mean pose collapse" problem
	- try applying SNR (signal-to-noise ratio) weighting to
	> De-emphasize high-noise timesteps & 
	> Force model to learn structure at low-noise timesteps
	- sol: `lambda_joints=0.05, lambda_vel=0.01, lambda_foot=0.02`; i.e. never 0s to prevent mean pose collapse from the start

- Run 2.Configs:
	1) snr: `cp, norm, cap=1.0`  --> not promising
	2) snr: `sq/lr, cap=1.0, rat=.05`  --> kinda promissing
	3) snr: `sq, cap=1.5`  --> better std growth than prev, but joint rotations are a long way to go
	4) snr:	`lr, cap=1.25` --> slower std growth than (1)
	5) snr: `sq/lr, cap=1.5, rat=.1`  --> 'sq' still performs better in std note alone

- Run 3.Misc Losses:
	1) `sq, cap=1.0`; `λ_tmp=1` added temporal --> poorer than with temporal
	2) `sq, cap=1.5, λ_tmp=.1`: `var_penalty = 1.0 / (var_pred over Frame Dim)` applied to L_simple --> exploding pred_std, max-abs of root-pos exceeding threshold of smpl FK for a scaled down version. where 
		```py 
		var_pred = m_pred.var(dim=1).mean(dim=-1) # var on S => (B, 1, D), mean on D => (B, 1))
		```
	3) `sq, cap=1.5, λ_tmp=.1`: `var_penalty = abs(log(var_pred) - log(var_gt))` applies to L_simple -->
		- actually looks like "its atleast trying", 
		- but too much movements on root and foot instead of focusing on rotations
	4) `sq, cap=1.5, λ_tmp=.1`: `scaled_penalty = 1.0 + 0.1 * clamp(var_penalty, max=1.0)` step wise bump applied additionally to L_simple after prev implementation --> very slow version above (3) not sure if it will get proper results as expected
	5) `sq, cap=1.5` : `, λ_div=.05,.1,.15 each 5 epoch` --> diversity varianve penality on axis angle rotations --> both version model (one with common film block, other with seperate film block params) suffer from occilating rotatory joints and root pos seems to static in motions like ballet dancing that involves actions with high movements, jumping, leg raises very high and stuff. collapses after 20 epochs back to static poses.

- Run 4.PTSS:
	- Start Config: `sq, cap=1, no tmp & div` - v1 & v2 `[lambdas: 0.05, 0.01, 0.025, 0, 0], lr=0.0001`
	- Epoch[1 -10] - `pts_prob = 0.2`
	- Epoch[11-15] - `pts_prob = 0.3`
	- Epoch[16-20] - `pts_prob = 0.4`
	- Epoch[21-25] - `pts_prob = 0.5`
	- so far steady decrease in all loss terms and increase in pred_std
	- Epoch[26-30] - `pts_prob = 0.2, sns_norm = True`; => jumpstart with lit high vals : good or bad? few SMPL FK triggers for root-pos scale more than threshold defined.
	- Epoch[31-35] - `pts_prob = 0.3`;
	- Epoch[36-40] - `pts_prob = 0.4, sns_norm = False, cap=2`;
	- Epoch[41-45] - `pts_prob = 0.5, cap=2`; 
	- Epoch[46-50] - `pts_prob = 0.6`, doubled lambdas --> `[lambdas: 0.1, 0.02, 0.05, 0, 0]`
	- Epoch[51-55] - `pts_prob = 0.7`
	- Epoch[56-60] - `pts_prob = 0.8, sns_norm = True`
	- improving but very very slow
	- Epoch[60-65] - `pts_prob = 0.2, cap=1, sns_norm = False`, lr increased --> 0.00025
	- in over conclusion: couldnt find significant differnce between v1 and v2, using v1 is better as no.of param is reduced in larger model configs

- Run 5.Big Model(d=256,n=8):
	- Start Config: `sq, cap=1, sns_norm = True` - v1 & v2 `[lambdas: 0.1, 0.02, 0.05, 0, 0], lr=0.0005`
	- added model state average techniquies to get to best convergence

### Optimized schedule targeting:
Training Loss - Target Values:

- L_joints : 0.01-0.02  ← Lower (better fit)
	- If L_joints < 0.01: May be overfitting
	- If L_joints > 0.03: Need more training or higher λ_joints

- L_vel	: 0.05-0.10   ← Higher (but motion present)
	- If        L_vel < 0.01    → Bad - Model predicting constant pose
	- If 0.05 < L_vel < 0.10 	→ Good - Dynamic motion with errors
	- If        L_vel > 0.15    → Needs work - Erratic/unsmooth motion

- L_foot : 0.0001-0.001 ← Slightly higher (real motion has some sliding)
	- If L_foot > 0.01: Too much sliding, increase λ_foot
	- If L_foot < 0.0001: Over-constrained, may hurt motion naturalness


---
---